# Limpieza de datos

La mayoría de la gente piensa que ser científico de datos significa estar corriendo modelos avanzados de Machine Learning todo el tiempo. La realidad es muy distinta:

![Tiempo de un científico de datos](./img/datascientist_time.jpeg)

La gran mayoría del tiempo se va **limpiando y organizando** los datos con los que queremos trabajar.

Los científicos de datos utilizamos herramientas como Pandas, Numpy, Matplotlib y Seaborn para limpiar los datos con los que queremos hacer algo.

## ETL

Un tipo de tarea que realizamos con gran frecuencia los científicos de datos son los **ETL**.

![Proceso de ETL](./img/etl.png)

ETL es un acrónimo que significa Extract, Transform, Load. Es un proceso que se utiliza para extraer datos de una fuente, transformarlos en un formato que sea adecuado para el análisis y cargarlos en una base de datos o algún otro sistema de almacenamiento.

## Ejemplo práctico

Los datos que usaremos para esta limpieza y nuestro siguiente análisis son datos de incidencia delictiva en nuestro país.

La iniciativa de datos abiertos del gobierno de México nos proporciona datos de incidencia delictiva desde 2015 hasta la fecha. Los datos se actualizan todos los meses y se pueden descargar desde el siguiente enlace: https://www.gob.mx/sesnsp/acciones-y-programas/datos-abiertos-de-incidencia-delictiva

---

Como podemos ver en el portal, se proporcionan los datos tanto a nivel estatal como a nivel municipal. En este caso, utilizaremos los datos a nivel estatal.

Descarguemos los datos y guardemos el archivo CSV en la carpeta data con el nombre `datos_delitos.csv`.

---

Ahora leamos el archivo CSV y veamos cómo se ven los datos.

Primero que nada, importemos pandas

Trabajo revisado y ejecutado con asistencia de IA, 21 de octubre de 2026, mediante IPython en el entorno del asistente. Se usan los archivos históricos del repositorio, no estadísticas actuales. Se corrigen filtros de subtipos; los datos estatales no identifican municipios. Las salidas anteriores se sustituyen por resultados de esta ejecución. melt reorganiza los valores; no los suma. La agregación conserva como NA los grupos enteramente ausentes.

In [1]:
import pandas as pd

In [2]:
# Error didáctico esperado: el archivo fuente no usa UTF-8.
try:
    pd.read_csv('data/datos_delitos.csv')
except UnicodeDecodeError:
    print('El archivo no es UTF-8; se utiliza Latin-1 tras revisar encabezados y texto.')

El archivo no es UTF-8; se utiliza Latin-1 tras revisar encabezados y texto.


Aquí tenemos un error muy común que suele ocurrir cuando un archivo se guarda en una computadora con cierto "encoding". 

Un encoding es una tabla que relaciona un número con un carácter. Por ejemplo, en la tabla ASCII, el número 65 corresponde a la letra "A".

Si el archivo que estamos leyendo fue guardado con un encoding distinto al que pandas espera, nos arrojará un error. 

Para solucionar esto, podemos utilizar el parámetro `encoding` de la función `pd.read_csv()` y especificar el encoding correcto.

¿Pero cómo sabemos con qué encoding cuenta el archivo?

La realidad es que la gran mayoría de los archivos los encontrarán en encoding utf-8 y Pandas no va a dar ningún error. Aquí estamos teniendo este problema porque la computadora que utilizan para generar este archivo uso un encoding diferente a utf-8.

En México, por lo general, si un archivo no está en utf-8, lo más seguro es que esté en `ISO-8859-1` o `latin1`.

Intentemos cargar el archivo especificando el encoding `ISO-8859-1`.

In [3]:
df = pd.read_csv('data/datos_delitos.csv', encoding='ISO-8859-1')
df.head()

Out[3]: 
    Año  Clave_Ent         Entidad  ... Octubre Noviembre Diciembre
0  2015          1  Aguascalientes  ...     2.0       2.0       1.0
1  2015          1  Aguascalientes  ...     0.0       0.0       1.0
2  2015          1  Aguascalientes  ...     0.0       0.0       0.0
3  2015          1  Aguascalientes  ...     0.0       0.0       0.0
4  2015          1  Aguascalientes  ...     0.0       0.0       0.0

[5 rows x 19 columns]


Y vemos que ya podemos leer correctamente el archivo.


¿Existe alguna forma de verificar el encoding de un archivo sin tener que estar adivinando?

ChatGTP generó el siguiente código:

In [4]:
import chardet

def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        rawdata = f.read()
    result = chardet.detect(rawdata)
    return result

file_path = './data/datos_delitos.csv'
encoding_info = detect_encoding(file_path)
print(f"Detected encoding: {encoding_info['encoding']}")

Detected encoding: Windows-1252


Sigamos...

In [5]:
df.tail(3)

Out[5]: 
        Año  Clave_Ent    Entidad  ... Octubre Noviembre Diciembre
31357  2024         32  Zacatecas  ...     NaN       NaN       NaN
31358  2024         32  Zacatecas  ...     NaN       NaN       NaN
31359  2024         32  Zacatecas  ...     NaN       NaN       NaN

[3 rows x 19 columns]


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31360 entries, 0 to 31359
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Año                     31360 non-null  int64  
 1   Clave_Ent               31360 non-null  int64  
 2   Entidad                 31360 non-null  object 
 3   Bien jurídico afectado  31360 non-null  object 
 4   Tipo de delito          31360 non-null  object 
 5   Subtipo de delito       31360 non-null  object 
 6   Modalidad               31360 non-null  object 
 7   Enero                   31360 non-null  int64  
 8   Febrero                 31360 non-null  int64  
 9   Marzo                   31360 non-null  int64  
 10  Abril                   31360 non-null  int64  
 11  Mayo                    31360 non-null  int64  
 12  Junio                   31360 non-null  int64  
 13  Julio                   28224 non-null  float64
 14  Agosto                  28224 non-null

Bien. Están muy bien los datos. Sin embargo, tenemos un problema con el que es muy común encontrarnos. 

Resulta que el formato en el que está el archivo es fácil entender por seres humanos:


```markdown
| Año      | Entidad  | Enero    | Febrero  | Mes X    |
|----------|----------|----------|----------|----------|
|   ...    |   ...    |   ...    |   ...    |   ...    |
|   ...    |   ...    |   ...    |   ...    |   ...    |
|   ...    |   ...    |   ...    |   ...    |   ...    |
|   ...    |   ...    |   ...    |   ...    |   ...    |
|   ...    |   ...    |   ...    |   ...    |   ...    |
```

Vemos que tenemos los meses como encabezados. Es decir, el archivo está tratando cada mes como si fuera una variable. 

Nosotros como científicos de datos estamos más interesados en conjuntos de datos que no estén en este formato de "resumen" o "tabla dinámica". Para nosotros, lo ideal sería que cada mes fuera simplemente una observación más en nuestro conjunto de datos. Es decir, queremos transformar la tabla de arriba en:

```markdown
| Año      | Entidad  | Mes        |
|----------|----------|------------|
|   ...    |   ...    |   Enero    |
|   ...    |   ...    |   Febrero  |
|   ...    |   ...    |   Marzo    |
|   ...    |   ...    |   Abril    |
|   ...    |   ...    |   Mes X    |
```

A este tipo de formato le llamos "formato largo de datos". 

Haremos las siguientes limpiezas:
* Transformar los nombres de las columnas para que no tengan caracteres especiales y estén siempre en minúsculas
* Convertir el dataset a un formato de datos "largo"

In [7]:
for col in df.columns:
    print(col)

Año
Clave_Ent
Entidad
Bien jurídico afectado
Tipo de delito
Subtipo de delito
Modalidad
Enero
Febrero
Marzo
Abril
Mayo
Junio
Julio
Agosto
Septiembre
Octubre
Noviembre
Diciembre


In [8]:
len(df.columns)

Out[8]: 19


### Limpiar nombres de columnas

In [9]:
def limpiar_columnas(df):
    columnas_limpias = []
    for col in df.columns:
        # convertir a minusculas, reemplazar espacios por guiones bajos y eliminar caracteres especiales
        col = col.lower().replace(" ", "_").replace("ñ", "ni").replace(".", "").replace("á", "a").replace("é", "e").replace("í","i").replace("ó", "o").replace("ú", "u")
        columnas_limpias.append(col)
    
    df.columns = columnas_limpias
    
    return df


In [10]:
df.head(2)

Out[10]: 
    Año  Clave_Ent         Entidad  ... Octubre Noviembre Diciembre
0  2015          1  Aguascalientes  ...     2.0       2.0       1.0
1  2015          1  Aguascalientes  ...     0.0       0.0       1.0

[2 rows x 19 columns]


In [11]:
df = limpiar_columnas(df)

In [12]:
df.head(3)

Out[12]: 
   anio  clave_ent         entidad  ... octubre noviembre diciembre
0  2015          1  Aguascalientes  ...     2.0       2.0       1.0
1  2015          1  Aguascalientes  ...     0.0       0.0       1.0
2  2015          1  Aguascalientes  ...     0.0       0.0       0.0

[3 rows x 19 columns]


Muy bien. Ahora lo que queremos hacer es quitar algunas columnas. Nos interesan nada más las siguientes:

In [13]:
df[['anio', 'entidad']]

Out[13]: 
       anio         entidad
0      2015  Aguascalientes
1      2015  Aguascalientes
2      2015  Aguascalientes
3      2015  Aguascalientes
4      2015  Aguascalientes
...     ...             ...
31355  2024       Zacatecas
31356  2024       Zacatecas
31357  2024       Zacatecas
31358  2024       Zacatecas
31359  2024       Zacatecas

[31360 rows x 2 columns]


In [14]:
df = df[['anio', 'clave_ent', 'entidad', 'tipo_de_delito', 'subtipo_de_delito', 'modalidad','enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio', 'julio', 'agosto', 'septiembre', 'octubre', 'noviembre', 'diciembre']]
df.head(2)

Out[14]: 
   anio  clave_ent         entidad  ... octubre noviembre diciembre
0  2015          1  Aguascalientes  ...     2.0       2.0       1.0
1  2015          1  Aguascalientes  ...     0.0       0.0       1.0

[2 rows x 18 columns]


### Formato largo de datos

Ahora, usaremos el método `melt` para convertir las columnas a observaciones.

Queremos convervar las coumnas:
* anio
* clave_ent
* entidad
* tipo_de_delito
* subtipo_de_delito
* modalidad

El resto de las columnas las vamos a juntar en una nueva columna llamada "nombre_mes" y sus valores los vamos a sumar en otra llamada "frecuencia"

In [15]:
print("Shape ", df.shape)

Shape  (31360, 18)


In [16]:
datos_long = df.melt(id_vars=['anio', 'clave_ent', 'entidad','tipo_de_delito', 'subtipo_de_delito', 'modalidad'], var_name='nombre_mes', value_name='frecuencia')
datos_detalle = datos_long.copy()  # Conservar subtipos para el análisis siguiente.

In [17]:
print("Shape: ", datos_long.shape)

Shape:  (376320, 8)


In [18]:
datos_long.head(5)

Out[18]: 
   anio  clave_ent         entidad  ...          modalidad nombre_mes frecuencia
0  2015          1  Aguascalientes  ...  Con arma de fuego      enero        3.0
1  2015          1  Aguascalientes  ...    Con arma blanca      enero        1.0
2  2015          1  Aguascalientes  ...  Con otro elemento      enero        0.0
3  2015          1  Aguascalientes  ...    No especificado      enero        2.0
4  2015          1  Aguascalientes  ...  Con arma de fuego      enero        0.0

[5 rows x 8 columns]


In [19]:
datos_long.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 376320 entries, 0 to 376319
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   anio               376320 non-null  int64  
 1   clave_ent          376320 non-null  int64  
 2   entidad            376320 non-null  object 
 3   tipo_de_delito     376320 non-null  object 
 4   subtipo_de_delito  376320 non-null  object 
 5   modalidad          376320 non-null  object 
 6   nombre_mes         376320 non-null  object 
 7   frecuencia         357504 non-null  float64
dtypes: float64(1), int64(2), object(5)
memory usage: 23.0+ MB


Supongamos que para este análisis, no nos importan los niveles subtipo de delito y modalidad. O sea, no queremos tener la distinción entre homicidios dolosos y culposos (sé que son bastante diferentes, pero simplifiquemos nuestro ejemplo).

Vamos a agrupar nuestro dataframe por anio, clave_ent, entidad, tipo_de_delito y nombre_mes. Esto hará que todos los tipos de homicidios se sumen al tipo "homicidio" o todos los tipos de robo de vehículo (con o sin violencia) se sumen a "robo de vehículo".

In [20]:
datos_long = datos_long.groupby(['anio', 'clave_ent', 'entidad', 'tipo_de_delito', 'nombre_mes'])['frecuencia'].sum(min_count=1).reset_index()

In [21]:
datos_long[datos_long.tipo_de_delito == 'Robo'].sample(15, random_state=42)

Out[21]: 
        anio  clave_ent  ... nombre_mes frecuencia
30146   2016         31  ...  diciembre      192.0
153513  2024         32  ...  noviembre        NaN
129026  2023         13  ...  diciembre      932.0
111270  2022          8  ...      junio     1190.0
76232   2019         31  ...       mayo      316.0
82946   2020         13  ...  diciembre      786.0
121825  2022         30  ...     agosto     1652.0
96384   2021          9  ...      abril     6572.0
107912  2022          1  ...       mayo      871.0
48865   2018          6  ...     agosto      729.0
70947   2019         20  ...      enero     1142.0
36867   2017         13  ...      enero     1144.0
54628   2018         18  ...    febrero      100.0
66628   2019         11  ...    febrero     3410.0
60869   2018         31  ...      julio      198.0

[15 rows x 6 columns]


Mostremos todos los estados y su respectiva clave

In [22]:
datos_long[['clave_ent', 'entidad']].drop_duplicates().sort_values('entidad')

Out[22]: 
       clave_ent                          entidad
0              1                   Aguascalientes
480            2                  Baja California
960            3              Baja California Sur
1440           4                         Campeche
2880           7                          Chiapas
3360           8                        Chihuahua
3840           9                 Ciudad de México
1920           5             Coahuila de Zaragoza
2400           6                           Colima
4320          10                          Durango
4800          11                       Guanajuato
5280          12                         Guerrero
5760          13                          Hidalgo
6240          14                          Jalisco
7200          16              Michoacán de Ocampo
7680          17                          Morelos
6720          15                           México
8160          18                          Nayarit
8640          19                       N

Ahora Veamos todos los datos de delitos de una entidad en específico. Por ejemplo, Nuevo león

In [23]:
datos_long[datos_long['clave_ent'] == 19].sample(15, random_state=42)

Out[23]: 
        anio  clave_ent  ...  nombre_mes frecuencia
24116   2016         19  ...        mayo       73.0
116170  2022         19  ...     octubre        9.0
100968  2021         19  ...       abril       18.0
100828  2021         19  ...     febrero       94.0
8648    2015         19  ...        mayo        7.0
24097   2016         19  ...      agosto      920.0
85907   2020         19  ...  septiembre        0.0
24331   2016         19  ...       marzo      244.0
116309  2022         19  ...       julio        2.0
54818   2018         19  ...   diciembre      610.0
101041  2021         19  ...      agosto        0.0
100986  2021         19  ...       junio      129.0
85521   2020         19  ...   noviembre        0.0
70474   2019         19  ...     octubre     1323.0
8867    2015         19  ...  septiembre      101.0

[15 rows x 6 columns]


In [24]:
datos_long[(datos_long['clave_ent'] > 19) &  (datos_long['clave_ent'] < 24) & (datos_long['tipo_de_delito'] == 'Homicidio')].sample(15, random_state=42)

Out[24]: 
        anio  clave_ent       entidad tipo_de_delito nombre_mes  frecuencia
25657   2016         22     Querétaro      Homicidio     agosto        30.0
133182  2023         22     Querétaro      Homicidio      junio        56.0
132226  2023         20        Oaxaca      Homicidio    octubre       165.0
87577   2020         23  Quintana Roo      Homicidio     agosto       112.0
132699  2023         21        Puebla      Homicidio      enero       108.0
56380   2018         22     Querétaro      Homicidio    febrero        30.0
87100   2020         22     Querétaro      Homicidio    febrero        38.0
118299  2022         23  Quintana Roo      Homicidio      enero       104.0
25661   2016         22     Querétaro      Homicidio      julio        40.0
147586  2024         20        Oaxaca      Homicidio    octubre         NaN
9345    2015         20        Oaxaca      Homicidio  noviembre       103.0
87581   2020         23  Quintana Roo      Homicidio      julio       129.0
14

### Valores de fechas

Finalmente, queremos tener una columna "fecha". Actualmente tenemos el año y el nombre del mes, pero no tenemos como tal una columna que tenga un tipo de dato fecha. Eso hace que filtrar por fecha sea complicado.

Por ejemplo, si queremos conocer todos los homicidios de Oaxaca en enero 2024, haríamos lo siguiente:

In [25]:
datos_long[
    (datos_long['clave_ent'] == 20) &
    (datos_long['tipo_de_delito'] == 'Homicidio') &
    (datos_long['anio'] == 2024) &
    (datos_long['nombre_mes'] == 'enero') 
]

Out[25]: 
        anio  clave_ent entidad tipo_de_delito nombre_mes  frecuencia
147579  2024         20  Oaxaca      Homicidio      enero       157.0


Creemos una columna de fecha. 

Primero tenemos que convertir el nombre de mes a un número, en donde 1 es enero, 2 febrero, etc.

In [26]:
datos_long.sample(2, random_state=42)

Out[26]: 
       anio  clave_ent  ... nombre_mes frecuencia
96514  2021         10  ...    octubre       46.0
78805  2020          5  ...     agosto        1.0

[2 rows x 6 columns]


In [27]:
datos_long["nueva_columna"] = "dato vacío"
datos_long.sample(6, random_state=42)

Out[27]: 
        anio  clave_ent  ... frecuencia nueva_columna
96514   2021         10  ...       46.0    dato vacío
78805   2020          5  ...        1.0    dato vacío
26229   2016         23  ...        1.0    dato vacío
129178  2023         14  ...       10.0    dato vacío
22843   2016         16  ...       50.0    dato vacío
65065   2019          8  ...      572.0    dato vacío

[6 rows x 7 columns]


In [28]:
# Diccionario de ayuda para convertir
meses = {
    "enero": 1,
    "febrero": 2,
    "marzo": 3,
    "abril": 4,
    "mayo": 5,
    "junio": 6,
    "julio": 7,
    "agosto": 8,
    "septiembre": 9,
    "octubre": 10,
    "noviembre": 11,
    "diciembre": 12
}

datos_long['mes'] = datos_long['nombre_mes'].map(meses)
datos_long.sample(5, random_state=42)

Out[28]: 
        anio  clave_ent               entidad  ... frecuencia nueva_columna  mes
96514   2021         10               Durango  ...       46.0    dato vacío   10
78805   2020          5  Coahuila de Zaragoza  ...        1.0    dato vacío    8
26229   2016         23          Quintana Roo  ...        1.0    dato vacío   11
129178  2023         14               Jalisco  ...       10.0    dato vacío   10
22843   2016         16   Michoacán de Ocampo  ...       50.0    dato vacío    3

[5 rows x 8 columns]


In [29]:
datos_long["frecuencia_mas_10"] = datos_long["frecuencia"] + 10
datos_long.sample(5, random_state=42)

Out[29]: 
        anio  clave_ent  ... mes frecuencia_mas_10
96514   2021         10  ...  10              56.0
78805   2020          5  ...   8              11.0
26229   2016         23  ...  11              11.0
129178  2023         14  ...  10              20.0
22843   2016         16  ...   3              60.0

[5 rows x 9 columns]


In [30]:
datos_long["anio_mes"] = datos_long["anio"].astype(str) + datos_long["mes"].astype(str)
datos_long.sample(6, random_state=42)

Out[30]: 
        anio  clave_ent               entidad  ... mes frecuencia_mas_10  anio_mes
96514   2021         10               Durango  ...  10              56.0    202110
78805   2020          5  Coahuila de Zaragoza  ...   8              11.0     20208
26229   2016         23          Quintana Roo  ...  11              11.0    201611
129178  2023         14               Jalisco  ...  10              20.0    202310
22843   2016         16   Michoacán de Ocampo  ...   3              60.0     20163
65065   2019          8             Chihuahua  ...   8             582.0     20198

[6 rows x 10 columns]


In [31]:
# yyyy-mm-dd, yy-mm-dd, yymmdd, yyyy/dd/mm

In [32]:
# Agregamos la columna de fecha juntando el año y el mes
datos_long['fecha'] = pd.to_datetime(datos_long['anio'].astype(str) + datos_long['mes'].astype(str), format='%Y%m')
datos_long.sample(5, random_state=42)

Out[32]: 
        anio  clave_ent  ... anio_mes      fecha
96514   2021         10  ...   202110 2021-10-01
78805   2020          5  ...    20208 2020-08-01
26229   2016         23  ...   201611 2016-11-01
129178  2023         14  ...   202310 2023-10-01
22843   2016         16  ...    20163 2016-03-01

[5 rows x 11 columns]


In [33]:
# Eliminamos las columnas que ya no necesitamos
datos_long = datos_long.drop(columns=['nueva_columna'])
datos_long.sample(5, random_state=42)

Out[33]: 
        anio  clave_ent  ... anio_mes      fecha
96514   2021         10  ...   202110 2021-10-01
78805   2020          5  ...    20208 2020-08-01
26229   2016         23  ...   201611 2016-11-01
129178  2023         14  ...   202310 2023-10-01
22843   2016         16  ...    20163 2016-03-01

[5 rows x 10 columns]


Veamos los homicidios en oaxaca de enero 2024 a la fecha

In [34]:
datos_long[
    (datos_long.tipo_de_delito == "Homicidio") &
    (datos_long.clave_ent == 20) &
    (datos_long.fecha >= '2024-01-01')
].sort_values('fecha')

Out[34]: 
        anio  clave_ent entidad  ... frecuencia_mas_10 anio_mes      fecha
147579  2024         20  Oaxaca  ...             167.0    20241 2024-01-01
147580  2024         20  Oaxaca  ...             180.0    20242 2024-02-01
147583  2024         20  Oaxaca  ...             200.0    20243 2024-03-01
147576  2024         20  Oaxaca  ...             218.0    20244 2024-04-01
147584  2024         20  Oaxaca  ...             246.0    20245 2024-05-01
147582  2024         20  Oaxaca  ...             212.0    20246 2024-06-01
147581  2024         20  Oaxaca  ...               NaN    20247 2024-07-01
147577  2024         20  Oaxaca  ...               NaN    20248 2024-08-01
147587  2024         20  Oaxaca  ...               NaN    20249 2024-09-01
147586  2024         20  Oaxaca  ...               NaN   202410 2024-10-01
147585  2024         20  Oaxaca  ...               NaN   202411 2024-11-01
147578  2024         20  Oaxaca  ...               NaN   202412 2024-12-01

[12 rows x 10 

In [35]:
datos_finales = datos_long[['anio', 'clave_ent', 'entidad', 'tipo_de_delito', 'nombre_mes', 'fecha', 'frecuencia']]
datos_finales.head(2)

Out[35]: 
   anio  clave_ent         entidad  ... nombre_mes      fecha frecuencia
0  2015          1  Aguascalientes  ...      abril 2015-04-01        0.0
1  2015          1  Aguascalientes  ...     agosto 2015-08-01        0.0

[2 rows x 7 columns]


Ya que tenemos muestros datos bien estructurados, los podemos guardar en nuestra computadora. Los guardaremos con el nombre "delitos.csv"

In [36]:
datos_finales.to_csv('data/delitos.csv', index=False)

In [37]:
datos_detalle['mes'] = datos_detalle['nombre_mes'].map(meses)
datos_detalle['fecha'] = pd.to_datetime(dict(year=datos_detalle['anio'], month=datos_detalle['mes'], day=1))
datos_detalle.to_csv('data/delitos_detalle.csv', index=False)
assert len(datos_detalle) == len(df) * 12
assert abs(datos_detalle.frecuencia.sum() - datos_finales.frecuencia.sum()) < 1e-8
assert datos_detalle.frecuencia.isna().any()
print('Formato largo:', datos_detalle.shape, 'Faltantes conservados:', datos_detalle.frecuencia.isna().sum())
print('Cobertura observada:', datos_detalle.loc[datos_detalle.frecuencia.notna(),'fecha'].min(), datos_detalle.loc[datos_detalle.frecuencia.notna(),'fecha'].max())

Formato largo: (376320, 10) Faltantes conservados: 18816
Cobertura observada: 2015-01-01 00:00:00 2024-06-01 00:00:00
